# PDF-Exact Two-Model Workflow

Stage 1 is the Task 8 coefficient DNN. Stage 2 is a coefficient-to-mask MLP using sigmoid-plus-MSE training. Stage 2 receives Stage 1 predicted coefficients for training, validation, and testing.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import torch
from config import Stage2ModelConfig, StageTrainingConfig, TwoStageRunConfig, TwoStageStackConfig
from pipeline import run_two_stage_once

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
stage2 = Stage2ModelConfig(
    hidden_layer_sizes=(512, 1024, 2048),
    dropout_rates=(0.2, 0.2, 0.2),
    model_type='mlp',
    use_rectangle_edge_weighting=False,
    use_foreground_pos_weight=False,
    training=StageTrainingConfig(
        epochs=150,
        batch_size=64,
        learning_rate=0.001,
        validation_frequency=30,
        verbose=True,
        early_stopping_patience=20,
        min_epochs=40,
        min_improvement=0.001,
        lr_drop_factor=0.5,
        lr_drop_period=50,
        weight_decay=0.0001,
        gradient_clip_norm=1.0,
        loss_type='mse',
    ),
)
run_config = TwoStageRunConfig(
    N=8,
    training_samples=10_000,
    validation_samples=2_000,
    test_samples=500,
    noise_sigma=0.01,
    seed=42,
    model=TwoStageStackConfig(stage2=stage2),
    output_dir=ROOT / 'outputs' / 'two_models_pdf_exact',
)
print(run_config)

In [ ]:
summary = run_two_stage_once(run_config, device=DEVICE)
print('Test IoU:', summary['metrics']['stage2_test']['mean_iou'])
print('Fixed IoU:', summary['metrics']['stage2_fixed']['mean_iou'])
print('Outputs:', run_config.run_output_dir)